In [ ]:
import pandas as pd 

In [43]:
df = pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [46]:
df.loc[14:24]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
14,다산콜센터,대중교통 안내,B2034,고객,7,버스요금,,Q,버스 요금은 얼마입니까?,,,,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
15,다산콜센터,대중교통 안내,B2034,상담사,8,,버스요금,A,,,,교통카드로 1200원 입니다.,"교통카드, 1200원",교통카드/결제수단/ 1200원/금액,"1200원,금액"
16,다산콜센터,대중교통 안내,B2035,고객,1,버스노선,,Q,수원에서 서울가는 버스노선을 알고싶습니다.,,,,"수원, 서울, 버스, 노선",수원/지명/ 서울/지명/ 버스/교통수단,"서울,교통수단"
17,다산콜센터,대중교통 안내,B2035,상담사,2,,버스노선,Q,,수원 어디에서 출발하십니까?,,,"수원, 출발",수원/지명/ 출발/출발지,"출발,출발지"
18,다산콜센터,대중교통 안내,B2035,고객,3,버스노선,,A,,,수원화성입니다.,,수원화성,수원화성/지명,"수원화성,지명"
19,다산콜센터,대중교통 안내,B2035,상담사,4,,버스노선,Q,,서울 어디로 가십니까?,,,서울,서울/지명,"서울,지명"
20,다산콜센터,대중교통 안내,B2035,고객,5,버스노선,,A,,,노량진으로 갑니다.,,노량진,노량진/동네,"노량진,동네"
21,다산콜센터,대중교통 안내,B2035,상담사,6,,버스노선,A,,,,수원화성에서 노량진으로 가는 노선은 7770번 버스를 타고 사당역 4번출구 정류장에...,"수원화성, 노량진, 노선, 버스, 사당, 역, 4번출구, 정류장",수원화성/지명/ 노량진/지명/ 버스/교통수단,"노량진,교통수단"
22,다산콜센터,대중교통 안내,B2035,고객,7,버스노선,,Q,지하철로 환승하는 길은 없습니까?,,,,지하철,지하철/교통수단,"지하철,교통수단"
23,다산콜센터,대중교통 안내,B2035,상담사,8,,버스노선,A,,,,"지하철로 환승하려면 7-1, 301, 5, 310 버스를 타고 수원역 정류장에서 1...","지하철, 버스, 수원, 역, 정류장",지하철/교통수단/ 버스/교통수단/ 수원/지명/ 역/역사,"버스,역사"


#### 문제 
1. 일반 행정 데이터와 대중교통 데이터를 로드 
2. 두개의 데이터를 단순 행 결합
3. 데이터의 필터링
    - 고객의 질문에서 즉각적으로 상담사의 답변이 오는 데이터들만 필터 
    - 고객의 질문이 존재 -> 다음 행의 상담사 답변이 존재하는가?
        - 비어있는 구간의 데이터의 형태를 일반화 
        - '' , ' ', '  ', ... :  '' 통일화 하려면? -> 텍스트 중간에 공백은 그대로 유지하고 좌우의 공백을 제거하는 함수 (strip())
4. 질문 중 중복 데이터를 제거 
5. 고객의 질문과 상담사의 답변이 하나의 행이 되도록 작업 
6. 완료가된 DataFrame을 저장 (민원 질의응답(즉답형데이터).csv) --> data의 백업 
5. 질문들을 모아서 토큰화(Komoran.morphs()) , 벡터화(TF-IDF) 작업 
6. 질문 목록 생성 
    - 여권 재발급 신청 방법을 알려주세요
    - 전입 신고가 인터넷으로 가능한가요?
    - 지방세 환급금을 어디서 신청하나요? 
    - 유사한 질문과 답변을 출력 (2개 씩)

In [ ]:
df2 = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")
df2.info()

In [ ]:
# 2개의 데이터를 단순한 행 결합 
# 인덱스를 기준으로 데이터를 필터링 하기 위해서 인덱스를 초기화
total_df = pd.concat([df, df2], axis=0, ignore_index=True)

In [ ]:
total_df.loc[0, ]

In [ ]:
# 비어있는 구간의 텍스트의 구조 확인 
total_df['고객질문(요청)'].value_counts()

In [ ]:
total_df = total_df.map(lambda x : str(x).strip())

In [ ]:
total_df['고객질문(요청)'].value_counts()

In [64]:
# 조건식 : 현재 행에서 고객질문(요청) 데이터가 '' 같지 않고 다음 행의 상담사답변이 '' 와 같지 않은 경우  
flag = (total_df['고객질문(요청)'] != '') & (total_df.shift(-1)['상담사답변'] != '') 
# 조건식2 : 현재 행에서 상담사답변이 ''와 같지 않고 전행의 고객질문(요청) 데이터가 ''와 같지 않은 경우 
flag2 = (total_df['상담사답변'] != '') & (total_df.shift(1)['고객질문(요청)'] != '')

total_df.loc[flag | flag2, ]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장
5,다산콜센터,대중교통 안내,B2033,상담사,6,,버스정류장,A,,,,문성초등학교 정류장에서 탑승하시면 됩니다.,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"
6,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
7,다산콜센터,대중교통 안내,B2033,상담사,8,,버스요금,A,,,,1200원 입니다.,1200원,1200원/금액,"1200원,금액"
8,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,,"서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89297,다산콜센터,일반행정 문의,B35809,상담사,16,,여성전용아파트,A,,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다.","입주, 신청서, 추천서, 급여내역서, 서류, 홈페이지","입주/이사, 신청서, 추천서, 급여내역서, 서류/문서, 홈페이지/웹페이지/인터넷","신청서,인터넷"
89298,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,,"입주, 순위","입주/이사, 순위/순서","순위,순서"
89299,다산콜센터,일반행정 문의,B35809,상담사,18,,여성전용아파트,A,,,,1~3순위가 있습니다.,순위,순위/순서,"순위,순서"
89300,다산콜센터,일반행정 문의,B35809,고객,19,여성전용아파트,,Q,1순위는 누가 되나요?,,,,순위,순위/순서,"순위,순서"


In [65]:
df3 = total_df.loc[flag, ]
df3['상담사답변'] = total_df.loc[flag2, '상담사답변'].values
df3

C:\Users\ekfla\AppData\Local\Temp\ipykernel_19580\1492766027.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['상담사답변'] = total_df.loc[flag2, '상담사답변'].values


,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,,정류장
6,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
8,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
10,다산콜센터,대중교통 안내,B2034,고객,3,버스시간,,Q,시간은 얼마정도 걸립니까?,,,약 1시간 10분정도 걸립니다.,시간,,시간
12,다산콜센터,대중교통 안내,B2034,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,서울역버스환승센터 정류장에서 탑승하시면 됩니다.,정류장,,정류장
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89292,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요?,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
89294,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요?,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
89296,다산콜센터,일반행정 문의,B35809,고객,15,여성전용아파트,,Q,신청은 어떻게 하나요?,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다.",신청,신청/접수,"신청,접수"
89298,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [66]:
df4 = total_df.loc[flag, ]
df4['상담사답변'] = total_df.shift(-1).loc[flag, '상담사답변'].values
df4

C:\Users\ekfla\AppData\Local\Temp\ipykernel_19580\562974741.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['상담사답변'] = total_df.shift(-1).loc[flag, '상담사답변'].values


,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,,정류장
6,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
8,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
10,다산콜센터,대중교통 안내,B2034,고객,3,버스시간,,Q,시간은 얼마정도 걸립니까?,,,약 1시간 10분정도 걸립니다.,시간,,시간
12,다산콜센터,대중교통 안내,B2034,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,서울역버스환승센터 정류장에서 탑승하시면 됩니다.,정류장,,정류장
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89292,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요?,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
89294,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요?,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
89296,다산콜센터,일반행정 문의,B35809,고객,15,여성전용아파트,,Q,신청은 어떻게 하나요?,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다.",신청,신청/접수,"신청,접수"
89298,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [67]:
# 파일로 저장 
df4.to_csv('민원 질의응답(즉답형데이터).csv', index = False)

In [68]:
df4.to_excel('민원 질의응답(즉답형데이터).xlsx', index = False)

In [69]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25445 entries, 4 to 89300
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        25445 non-null  object
 1   카테고리       25445 non-null  object
 2   대화셋일련번호    25445 non-null  object
 3   화자         25445 non-null  object
 4   문장번호       25445 non-null  object
 5   고객의도       25445 non-null  object
 6   상담사의도      25445 non-null  object
 7   QA         25445 non-null  object
 8   고객질문(요청)   25445 non-null  object
 9   상담사질문(요청)  25445 non-null  object
 10  고객답변       25445 non-null  object
 11  상담사답변      25445 non-null  object
 12  개체명        25445 non-null  object
 13  용어사전       25445 non-null  object
 14  지식베이스      25445 non-null  object
dtypes: object(15)
memory usage: 3.1+ MB


In [70]:
# 질문중 중복 질문에 대한 제거 
df4.drop_duplicates('고객질문(요청)', inplace=True)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_19580\2752480615.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4.drop_duplicates('고객질문(요청)', inplace=True)


In [71]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18948 entries, 4 to 89300
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        18948 non-null  object
 1   카테고리       18948 non-null  object
 2   대화셋일련번호    18948 non-null  object
 3   화자         18948 non-null  object
 4   문장번호       18948 non-null  object
 5   고객의도       18948 non-null  object
 6   상담사의도      18948 non-null  object
 7   QA         18948 non-null  object
 8   고객질문(요청)   18948 non-null  object
 9   상담사질문(요청)  18948 non-null  object
 10  고객답변       18948 non-null  object
 11  상담사답변      18948 non-null  object
 12  개체명        18948 non-null  object
 13  용어사전       18948 non-null  object
 14  지식베이스      18948 non-null  object
dtypes: object(15)
memory usage: 2.3+ MB


In [72]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vec = TfidfVectorizer(
    tokenizer=tokenize, 
    lowercase=False, 
    ngram_range=(1,1), 
    min_df = 5, 
    max_df=0.8
)

In [74]:
# 고객질문(요청) 데이터를 벡터화 
X = vec.fit_transform(
    df4['고객질문(요청)']
)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [75]:
new_questions = [
    '여권 재발급 신청 방법을 알려주세요', 
    '전입 신고가 인터넷에서 가능한가요?', 
    '지방세 환급금을 어디서 신청하나요?'
]

In [76]:
test = vec.transform(new_questions)

In [77]:
sims = cosine_similarity(test, X)

In [78]:
sims

array([[0.        , 0.        , 0.03768027, ..., 0.        , 0.        ,
        0.        ],
       [0.10382223, 0.        , 0.08867365, ..., 0.        , 0.        ,
        0.05229653],
       [0.02110115, 0.        , 0.03995857, ..., 0.03382659, 0.0321687 ,
        0.02175141]], shape=(3, 18948))

In [79]:
df4.reset_index(drop=True, inplace=True)

In [80]:
for idx, sim in enumerate(sims):
    question = new_questions[idx]

    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        print(f"""
            유사도의 값 : {round(sim[i], 3)}
            고객의 질문 : {question}
            유사 질문 : {df4.loc[i, '고객질문(요청)']}
            답변 : {df4.loc[i, '상담사답변']}
        """)


            유사도의 값 : 0.707
            고객의 질문 : 여권 재발급 신청 방법을 알려주세요
            유사 질문 : 신청방법을 알려주세요.
            답변 : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
        

            유사도의 값 : 0.655
            고객의 질문 : 여권 재발급 신청 방법을 알려주세요
            유사 질문 : 신청방법 좀 알려주세요?
            답변 : 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
        

            유사도의 값 : 0.62
            고객의 질문 : 전입 신고가 인터넷에서 가능한가요?
            유사 질문 : 인터넷으로도 신고가능한가요?
            답변 : 방문 접수밖에 안됩니다.
        

            유사도의 값 : 0.583
            고객의 질문 : 전입 신고가 인터넷에서 가능한가요?
            유사 질문 : 전입신고는 가서 해야되죠?
            답변 : 방문신고는 신 거주지 동주민센터에서만 가능합니다.
        

            유사도의 값 : 0.76
            고객의 질문 : 지방세 환급금을 어디서 신청하나요?
            유사 질문 : 지방세 환급금 신청은 어떻게 해야하죠?
            답변 : 인터넷에서 접수를 하셔야 합니다
        

            유사도의 값 : 0.566
            고객의 질문 : 지방세 환급금을 어디서 신청하나요?
            유사 질문 : 환급금을 기부할 수도 있나요?
            답변 : 네 환급금을 사회복지공동모

- 고객 질문의 데이터를 이용해서 카테고리를 분류하는 모델을 생성 
    - 고객 질문 데이터를 이용하여 토큰화, 백터화 (독립 변수)
    - 카테고리 일반 행정, 대중 교통을 타겟 데이터
        - 카테고리 데이터를 LabelEncoder를 이용하여 수치화 변환
    - SVC 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습 
    - new_questions의 카테고리들을 확인 
- 예측이 된 카테고리를 이용하여 df4에서 카테고리로 필터링 
- new_qustions를 벡터화하여 유사도를 확인 

In [82]:
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

In [83]:
X = df4['고객질문(요청)'].values
y = df4['카테고리'].values

In [84]:
# 독립 변수 벡터화
X_vec = vec.fit_transform(X)
# 종속 변수 LabelEncoder
le = LabelEncoder()
y_le = le.fit_transform(y)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [85]:
svc = SVC(
    C = 1.0, 
    kernel= 'linear', 
    random_state=42
)

In [86]:
svc.fit(X_vec, y_le)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [87]:
new_vec = vec.transform(new_questions)
pred = svc.predict(new_vec)

In [88]:
pred

array([1, 1, 1])

In [89]:
# 라벨링이 된 데이터를 원본의 형태로 변환 
le.inverse_transform(pred)

array(['일반행정 문의', '일반행정 문의', '일반행정 문의'], dtype=object)

In [92]:
new_questions = [
    '여권 재발급 신청 방법을 알려주세요', 
    '전입 신고가 인터넷으로 가능한가요', 
    '서울역에서 영등포로 가려면 어떻게 가나요', 
    '강북구청 근처에서 가장 가까운 정류장을 어디인가요'
]

In [93]:
new_vec = vec.transform(new_questions)
pred2 = svc.predict(new_vec)

In [95]:
pred2_cate = le.inverse_transform(pred2)

In [97]:
for vec_data, cate in zip(new_vec, pred2_cate):
    # 새로운 질문의 예측된 카테고리를 이용하여 필터링을 하고 벡터화 
    x = vec.transform(
        df4.loc[ df4['카테고리'] == cate, '고객질문(요청)' ]
    )
    # 벡터화된 x와 vec_data를 기준으로 코사인 유사도를 계산 : 질문이 1개이기때문에 sims를 1차원으로 변환
    sims = cosine_similarity(vec_data, x).ravel()
    # 유사도를 내림차순 정렬로 인덱스의 값들을 확인 
    idxs = sims.argsort()[::-1]
    for idx in idxs[:2]:
        print(f"유사도 : {round( sims[idx], 3 )}")
        print(
            "유사 질문 :" , df4.loc[df4['카테고리'] == cate, '고객질문(요청)'].iloc[idx]
        )
        print(
            "답변 : ", df4.loc[ df4['카테고리'] == cate, '상담사답변' ].iloc[idx]
        )
    

유사도 : 0.707
유사 질문 : 신청방법을 알려주세요.
답변 :  주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사도 : 0.655
유사 질문 : 신청방법 좀 알려주세요?
답변 :  우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
유사도 : 0.584
유사 질문 : 인터넷으로도 신고가능한가요?
답변 :  방문 접수밖에 안됩니다.
유사도 : 0.526
유사 질문 : 인터넷으로 가능한가요?
답변 :  인터넷으로 신청 가능합니다.
유사도 : 0.671
유사 질문 : 서울역에서 발산역 지하철로어떨게 가나요?
답변 :  서울역에서 공항철도지하철을 이용하셔서 김포공항에 내리시고 5호선환승하셔서 발산역에서 내리시면됩니다
유사도 : 0.571
유사 질문 : 어떻게 가나요?
답변 :  오류역에서 지하철을 탄 후 대전역 지하철에서 내리신 후 14번 버스를 타면됩니다.
유사도 : 0.521
유사 질문 : 그럼 가장 가까운 지하철역이 어디인가요?
답변 :  신분당선 광교중앙역입니다.
유사도 : 0.507
유사 질문 : 옮긴 정류장이 어딘가요?
답변 :  ㅇㅇ방향으로 ㅇㅇ미터 걸어가시면 있습니다
